# ПР-10. Кластеризация и сегментация

Применить K-means без целевой разметки, выбрать разумное число групп и описать кластеры через исходные признаки.

**Данные:** см. README проекта. Выполняйте ячейки сверху вниз; фиксируйте результаты в отчёте.

## 1. Подготовка данных

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
ROOT=Path.cwd()
if not (ROOT/'data').exists() and (ROOT.parent/'data').exists(): ROOT=ROOT.parent
df=pd.read_csv(ROOT/'data'/'tips_dirty.csv')
df.head(), df.isna().sum()

## 1. Подготовка и подбор k

Используем числовые признаки счёта, чаевых и размера компании. Стандартизатор нужен, чтобы суммы не доминировали над количеством гостей.

In [ ]:
features=['total_bill','tip','size']
work=df[features].dropna().copy()
scaler=StandardScaler(); Z=scaler.fit_transform(work)
ks=range(2,8)
inertias=[]; silhouettes=[]
for k in ks:
    km=KMeans(n_clusters=k,random_state=42,n_init=20).fit(Z)
    inertias.append(km.inertia_); silhouettes.append(silhouette_score(Z,km.labels_))
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].plot(list(ks),inertias,marker='o'); ax[0].set_title('Elbow')
ax[1].plot(list(ks),silhouettes,marker='o'); ax[1].set_title('Silhouette')
plt.show()

## 2. Описание групп

Выберите k по графикам и удобству содержательной интерпретации. Силуэт сам по себе не гарантирует полезной сегментации.

In [ ]:
chosen_k=3  # измените после анализа графиков
km=KMeans(n_clusters=chosen_k,random_state=42,n_init=20)
work['cluster']=km.fit_predict(Z)
summary=work.groupby('cluster')[features].agg(['count','mean','median']).round(2)
display(summary)
print('silhouette:',round(silhouette_score(Z,work['cluster']),3))
work['cluster'].value_counts().sort_index()

## Контрольные вопросы и вывод

1. Как масштабирование повлияло на расстояния?
2. Какой k выбрали и какие свидетельства использовали?
3. Опишите каждый кластер словами по средним/медианным чеку, чаевым и размеру группы.
4. Почему высокий silhouette не является единственным критерием качества сегментации?

**В отчёте:** графики elbow/silhouette, выбранное k, таблица характеристик и словесное описание каждого кластера.